# Tools in DemoGPT

DemoGPT AgentHub provides **12+ built-in tools** and the ability to create **custom ones**. Tools are the building blocks that give agents the ability to interact with the outside world -- searching the web, running code, fetching URLs, and more.

All tools extend `BaseTool` and implement a `run()` method. This simple interface makes it easy to understand existing tools and create your own.

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## Search Tools

DemoGPT includes several tools for searching the web, Wikipedia, and academic papers.

### TavilySearchTool

Search the web using Tavily.

In [ ]:
from demogpt_agenthub.tools import TavilySearchTool

tool = TavilySearchTool(max_results=4)
result = tool.run("What is artificial intelligence?")
print(result)

### WikipediaTool

Search and retrieve summaries from Wikipedia articles.

In [ ]:
from demogpt_agenthub.tools import WikipediaTool

tool = WikipediaTool()
result = tool.run("Machine Learning")
print(result)

### ArxivTool

Search academic papers on arXiv.

In [ ]:
from demogpt_agenthub.tools import ArxivTool

tool = ArxivTool()
result = tool.run("large language models")
print(result)

## Data Processing Tools

These tools let agents run code, execute shell commands, and fetch web content.

### PythonTool

Execute Python code snippets and capture the output.

In [ ]:
from demogpt_agenthub.tools import PythonTool

tool = PythonTool()
result = tool.run("import math\nprint(math.sqrt(144))")
print(result)

### BashTool

Execute shell commands directly.

In [ ]:
from demogpt_agenthub.tools import BashTool

tool = BashTool()
result = tool.run("echo 'Hello from BashTool!' && date")
print(result)

### RequestUrlTool

Fetch content from a URL.

In [ ]:
from demogpt_agenthub.tools import RequestUrlTool

tool = RequestUrlTool()
result = tool.run("https://httpbin.org/get")
print(result)

## Specialized Tools

These tools provide access to specific services and APIs.

### WeatherTool

Get current weather data for any city. Requires an `OPENWEATHERMAP_API_KEY` environment variable.

In [ ]:
import os
# os.environ["OPENWEATHERMAP_API_KEY"] = "your-api-key-here"

from demogpt_agenthub.tools import WeatherTool

tool = WeatherTool()
result = tool.run("London")
print(result)

### YouTubeSearchTool

Search for videos on YouTube.

In [ ]:
from demogpt_agenthub.tools import YouTubeSearchTool

tool = YouTubeSearchTool()
result = tool.run("Python programming tutorial")
print(result)

### StackOverFlowTool

Search StackOverflow for programming questions and answers.

In [ ]:
from demogpt_agenthub.tools import StackOverFlowTool

tool = StackOverFlowTool()
result = tool.run("How to read a file in Python")
print(result)

## Creating Custom Tools

You can create your own tools by extending `BaseTool`. Every custom tool needs:

- A `name` attribute: a short identifier for the tool
- A `description` attribute: tells agents when and how to use the tool
- A `run()` method: contains the tool's logic and returns a string result

In [ ]:
from demogpt_agenthub.tools import BaseTool

class CalculatorTool(BaseTool):
    def __init__(self):
        self.name = "CalculatorTool"
        self.description = "Performs basic arithmetic calculations. Input should be a mathematical expression like '2 + 3' or '10 * 5'"
        super().__init__()
    
    def run(self, expression: str):
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

calc = CalculatorTool()
print(calc.run("25 * 4 + 10"))

## Using Custom Tools with Agents

Custom tools become truly powerful when combined with agents. A `ToolCallingAgent` can automatically decide which tool to use based on the user's query.

In [ ]:
from demogpt_agenthub.agents import ToolCallingAgent
from demogpt_agenthub.llms import OpenAIChatModel
from demogpt_agenthub.tools import BaseTool

# Define the custom tool
class CalculatorTool(BaseTool):
    def __init__(self):
        self.name = "CalculatorTool"
        self.description = "Performs basic arithmetic calculations. Input should be a mathematical expression like '2 + 3' or '10 * 5'"
        super().__init__()
    
    def run(self, expression: str):
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

# Create the LLM and agent
llm = OpenAIChatModel(model="gpt-4o-mini")
agent = ToolCallingAgent(llm=llm, tools=[CalculatorTool()])

# The agent will automatically use the CalculatorTool
result = agent.run("What is 125 * 8 + 37?")
print(result)

## Tool Properties

Every tool in DemoGPT has two key attributes that agents rely on:

- **`name`**: A short, unique identifier for the tool (e.g., `"TavilySearchTool"`, `"PythonTool"`). Agents use this to reference which tool they want to call.
- **`description`**: A natural language explanation of what the tool does and what kind of input it expects. This is critical -- agents read the description to decide **which tool is most appropriate** for a given task.

When you create custom tools, writing a clear and specific `description` is essential. A vague description may cause the agent to misuse the tool or skip it entirely.

## Summary

In this notebook we covered:

- **Search Tools**: `TavilySearchTool`, `WikipediaTool`, and `ArxivTool` for retrieving information from the web and academic sources
- **Data Processing Tools**: `PythonTool`, `BashTool`, and `RequestUrlTool` for executing code and fetching web content
- **Specialized Tools**: `WeatherTool`, `YouTubeSearchTool`, and `StackOverFlowTool` for domain-specific tasks
- **Custom Tools**: How to extend `BaseTool` to create your own tools with a `name`, `description`, and `run()` method
- **Agent Integration**: How to pass custom tools to a `ToolCallingAgent` so the agent can automatically select and use them

All tools follow the same simple pattern: extend `BaseTool`, set `name` and `description`, implement `run()`. This makes the tool system easy to learn and extend.